[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/2_Robot_Chasing/code/baseline/solution.ipynb)

# Robot Chasing — baseline on Google Colab

Run all (GPU runtime recommended). The first cell fetches the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-robot-chasing) and recreates the contest file layout; every cell after it is the original, untouched baseline.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install huggingface_hub')

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-robot-chasing", repo_type="dataset"))
def link(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src, dst)
# merged dataset root: split folders + *_answers files, public and private together
DSROOT = Path("/content/dsroot").resolve()
for sub in ("public","private"):
    d = DATA/sub
    if d.is_dir():
        for child in d.iterdir():
            link(child, DSROOT/child.name)
print("dataset root:", DSROOT, "->", sorted(p.name for p in DSROOT.iterdir()))

EVAL_SPLIT = "test_leaderboard_a"
ws = Path.cwd()
for name in ("train","pretrain","pretest"): link(DSROOT/name, ws/"dataset"/name)
link(DSROOT/"train_answers.json", ws/"dataset"/"train_answers.json")
link(DSROOT/EVAL_SPLIT, ws/"dataset"/"test_public")
print("ready: dataset/ (test_public ->", EVAL_SPLIT + ")")


# Catch the Robot — always up baseline

Predict absolute action `0=up` for every observation.

In [ ]:
import json, os
from pathlib import Path

root = Path(os.environ.get("DATASET_ROOT", "dataset"))
observations = json.load(open(root / "test_public" / "observations.json"))

# Absolute actions: 0=up, 1=down, 2=left, 3=right, 4=pickup, 5=drop.
predictions = [0 for _ in observations]

with open("predictions.json", "w") as handle:
    json.dump(predictions, handle)
print("wrote", len(predictions), "predictions to predictions.json")